# The asteroidea notebook: here is the data

This notebook is the shortest path from the Python SDK, `asteroidea`, to the platform's data. It follows the
web app's views, one short section each:

1. sign in,
2. the studies,
3. the participants with both MRI and electrophysiology, and the links that join their subject labels,
4. one session's units and their brain regions,
5. that session's response heatmap (every unit's z-scored rate around scene cuts, grouped by region).

Every step reads the catalog or a summary the platform has already computed and stored. Nothing opens a raw
recording, so **the main path (sections 1 to 5) runs in about a minute or less**, most of it the first
catalog read. A last, optional section opens the raw NWB file; it is slow and off by default.

**Before you start.** You need a `platform.json` from the platform administrator or from `make researcher EMAIL=<you>`;
it lives in `~/.config/asteroidea/`. The setup command (or `make notebook`) builds the Python environment
this notebook runs in.

## 1. Sign in

Everything this notebook imports: the SDK, numpy, pandas, matplotlib and IPython (`display`, imported as
`show`), plus the standard library. `asteroidea.sign_in()` prints a login address the first time. Open it in a
browser, sign in, and paste the address the browser ends on into the box below the cell. Later runs reuse the
saved sign-in and print nothing.

In [ ]:
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display as show

import asteroidea

In [ ]:
asteroidea.sign_in()

## 2. Studies

A study is one piece of published work; its datasets are where its files came from (a DANDI dandiset, an
OpenNeuro dataset). `studies()` returns one row per study and dataset, with the dataset's role and tier.

In [ ]:
studies = asteroidea.studies().to_pandas()
studies[["study", "dataset", "role", "tier", "title"]]

## 3. Participants with MRI and electrophysiology

A participant is one person, named by a `subject_key`, possibly with a different subject label in each
dataset. `participants(has=[...])` keeps the people who have every listed modality: here sorted units
(`units`) and functional MRI (`func`).

In [ ]:
both = asteroidea.participants(has=["units", "func"]).to_pandas()
both

Patients 41 and 62 of the movie-watching study are `sub-CS41` and `sub-CS62` in DANDI 000623 and `sub-p41cs`
and `sub-p62cs` in OpenNeuro ds004798. A **link** joins the two labels under one `subject_key`. These links
are `pending`: the curator proposed them from the datasets' own cross-references, and no one has
confirmed them yet. `links()` shows each one with who linked it and why.

In [ ]:
PATIENTS = ("sub-CS41", "sub-CS62")
links = asteroidea.links(study="movie-watching").to_pandas()
keys = links.loc[links["subject"].isin(PATIENTS), "subject_key"]
links[links["subject_key"].isin(keys)][
    ["subject_key", "dataset", "subject", "status", "linked_by", "basis"]
]

## 4. One session's units

`session_summary()` reads what the web app's session page draws: the platform computed it once and stored
it, so this is a read, not a computation. The session is patient 62's second recording (`ses-P62CSR2`).
The units table has one row per sorted neuron, with the brain region of its electrode (`location`).

In [ ]:
SESSION = ("dandi-000623", "sub-CS62", "ses-P62CSR2")
summary = asteroidea.session_summary(*SESSION)
if summary is None:
    msg = (
        f"No stored summary for {'/'.join(SESSION)} yet. The platform builds one per "
        "session; until then the web app's session page computes it live."
    )
    raise LookupError(msg)
units = summary.units.to_pandas()
show(
    units.groupby("location", dropna=False)
    .size()
    .rename("units")
    .reset_index()
    .sort_values("units", ascending=False)
)
units[["unit_index", "location", "spike_count", "mean_rate_hz"]]

The events the summary was aligned on, as the dataset's event recipe resolves them: one row per event, its
onset in seconds and its attributes. `averaging` says whether a type is averaged over its events (scene cuts,
recognition onsets) or happens once (the movie onset).

In [ ]:
events = summary.events.to_pandas()
events.groupby(["event_type", "averaging"]).size().rename("events").reset_index()

## 5. The response heatmap

For each averaged event type the summary holds every unit's firing rate in 50 ms bins from 1 s before to 2 s
after the event, and its z-score against the unit's own baseline: **baseline −1 to 0 s, z**. A z of 0 is the
unit's usual pre-event rate; +2 is two standard deviations above it. A unit whose baseline never varies has
no z and is left out of the image (it is listed instead).

The rows are grouped by region, as on the web page, and within a region sorted by the mean z over the
response window (highest first). The numbers are an **exploratory screen**, not a finished analysis.

In [ ]:
EVENT = "scene_cut" if "scene_cut" in summary.rates else next(iter(summary.rates))
rates = summary.rates[EVENT]
edges = np.asarray(rates.edges)
window = rates.defaults.get("response_window_s", [0.0, 1.0])
in_window = (edges[:-1] >= window[0]) & (edges[1:] <= window[1])
location = dict(zip(units["unit_index"], units["location"], strict=True))
rows = pd.DataFrame(
    {
        "unit_index": rates.units.column("unit_index").to_pylist(),
        "z": rates.units.column("z").to_pylist(),
    }
)
rows["location"] = rows["unit_index"].map(location).fillna("no location")
no_baseline = rows[rows["z"].isna()]
rows = rows[rows["z"].notna()].copy()
rows["mean_z"] = [float(np.mean(np.asarray(z)[in_window])) for z in rows["z"]]
rows = rows.sort_values(["location", "mean_z"], ascending=[True, False])
show(
    pd.DataFrame(
        {
            "event type": [EVENT],
            "events": [rates.n_trials],
            "units drawn": [len(rows)],
            "units without baseline variance": [len(no_baseline)],
            "label": [rates.label],
        }
    )
)

In [ ]:
Z_RANGE = 3
image = np.vstack([np.asarray(z) for z in rows["z"]])
fig, ax = plt.subplots(figsize=(8, max(4, len(rows) * 0.12)))
drawn = ax.imshow(
    image,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-Z_RANGE,
    vmax=Z_RANGE,
    extent=(edges[0], edges[-1], len(rows), 0),
    interpolation="nearest",
)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
starts = rows.reset_index(drop=True).groupby("location", sort=False).indices
for at in starts.values():
    ax.axhline(at[0], color="black", linewidth=0.6)
ax.set_yticks(
    [at[0] + len(at) / 2 for at in starts.values()],
    [f"{name} ({len(at)})" for name, at in starts.items()],
)
ax.set_xlabel(f"time from {EVENT} (s)")
ax.set_ylabel("region (units)")
ax.set_title(f"{'/'.join(SESSION)}: response to {EVENT} (n = {rates.n_trials})")
fig.colorbar(drawn, ax=ax, label="baseline \N{MINUS SIGN}1 to 0 s, z")
fig.tight_layout()

The responsive screen the platform ran with the same events: per unit, the permutation p and the
Benjamini-Hochberg q across units, and `screened` when q is under the defaults' alpha. The most responsive
units first.

In [ ]:
rates.responsive.to_pandas().sort_values("q").head(10)

## Going deeper (slow): open the raw NWB

Everything above came from the stored summary. The same numbers come from the recording itself, and the SDK
opens it lazily: `open_nwb` reads only the byte ranges your code touches, over the network. That is slower
(tens of seconds for the first read of a 1 GB file), so this section is **off by default**. Set `GO_DEEPER`
to `True` and run the two cells below to read the spike times of the unit at the top of the heatmap and draw
its raster around the same events.

In [ ]:
GO_DEEPER = False

In [ ]:
if GO_DEEPER:
    top = int(rows.iloc[0]["unit_index"])
    nwb: Any = asteroidea.open_nwb(summary.asset_id)
    spikes = np.asarray(nwb.units["spike_times"][top], dtype=float)
    onsets = events.loc[events["event_type"] == EVENT, "onset_s"].to_numpy()
    fig, ax = plt.subplots(figsize=(7, 4))
    for trial, onset in enumerate(onsets):
        near = spikes[(spikes >= onset + edges[0]) & (spikes < onset + edges[-1])]
        ax.scatter(near - onset, np.full(near.size, trial), s=2, color="black")
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel(f"time from {EVENT} (s)")
    ax.set_ylabel("event")
    ax.set_title(f"Unit {top} ({location.get(top)}), read from the raw NWB file")
    fig.tight_layout()
    show(pd.DataFrame({"bytes fetched": [nwb.remote_file.bytes_fetched]}))